# 03 — Train MULTIVARIATE AE & upload to KQL `models`

Multivariate sibling of `02_train_univariate_ae.ipynb`:

1. Read 8 sensors of **one machine** from the wide MV `raw_telemetry_wide_mv` (forward-filled).
2. Build **sliding** windows of `WINDOW_SIZE` bins over an `(T, 8)` matrix.
3. Train a small LSTM autoencoder in PyTorch.
4. Export a `ScoreWrapper` to **ONNX** (output = MSE per window across all sensors).
5. Compute `threshold = mean(loss) + 3 * std(loss)` on the training set and store it in `metadata.threshold`.
6. Upload to the `models` table as a new `version` of `multivariate_ae__M-001`.
7. Smoke-test by calling `score_multivariate_onnx_batch_from_mv(...)` directly in KQL.

Prereqs: `kql/05_multivariate_mv.kql` already applied (MV exists and is being maintained), simulator running, `python` plugin enabled.

Runs both **locally** (`.venv`, device-code auth) and **inside Fabric** (auto-detected via `notebookutils`).


## 0. Install runtime dependencies (Fabric-only)

In Microsoft Fabric the Spark Python runtime ships with a curated package set
that does not include azure-kusto-data. The cell below installs the SDK on
first run; on a local .venv it is a no-op. For repeated runs in production,
prefer attaching a Fabric Environment with these libraries pre-published.


In [ ]:
# Fabric runtime does not preinstall azure-kusto-data; install it on first run.
# %pip restarts the Python kernel, so this MUST be the first executed cell.
# On a local .venv these packages are already present and pip is a no-op.
%pip install -q azure-kusto-data azure-identity python-dotenv


## 1. Config

In [1]:
MACHINE      = "M-001"
# IMPORTANT: order MUST match `pack_array(...)` in build_multivariate_windows_batch_from_mv
# (file: kql/05_multivariate_mv.kql).
SENSORS      = [
    "temperature_motor",
    "temperature_bearing",
    "vibration_axial",
    "vibration_radial",
    "current",
    "power",
    "spindle_rpm",
    "pressure_hydraulic",
]
WINDOW_SIZE  = 64           # 64 bins of 1s = 64s per window
BIN_SIZE     = "1s"
LOOKBACK     = "30m"        # KQL timespan literal for training data
EPOCHS       = 100          # mini-batch SGD with cosine LR; ~35 steps/epoch * 100 = 3500 updates
BATCH        = 32
LR           = 1e-3
HIDDEN       = 64
MODEL_NAME   = f"multivariate_ae__{MACHINE}"
THRESHOLD_K  = 4.0          # threshold = mean(loss) + K * std(loss); K=4 on normalized features keeps FPR low

## 2. Auth + Kusto client (dual mode: local `.venv` or Fabric notebook)

In [2]:
import base64
import datetime as dt
import io
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table

FABRIC_API   = "https://api.fabric.microsoft.com/v1"
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"

try:
    import notebookutils  # only present inside Fabric runtime
    IN_FABRIC = True
except ImportError:
    IN_FABRIC = False

if IN_FABRIC:
    from azure.identity import DefaultAzureCredential
    ctx        = notebookutils.runtime.context
    WORKSPACE  = ctx.get("currentWorkspaceName")
    TENANT_ID  = ctx.get("tenantId", "")
    KQLDB_NAME = os.environ.get("FABRIC_KQLDB_NAME", "kql_telemetry")
    cred       = DefaultAzureCredential()
    print(f"[auth] Fabric runtime (workspace={WORKSPACE})")
else:
    from dotenv import load_dotenv
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / ".env").exists() and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent
    load_dotenv(REPO_ROOT / ".env")
    sys.path.insert(0, str(REPO_ROOT / "tools"))
    from _fabric_auth import get_credential  # noqa: E402
    TENANT_ID  = os.environ["FABRIC_TENANT_ID"]
    WORKSPACE  = os.environ["FABRIC_WORKSPACE_NAME"]
    KQLDB_NAME = os.environ["FABRIC_KQLDB_NAME"]
    cred       = get_credential(TENANT_ID, FABRIC_SCOPE, REPO_ROOT)
    print(f"[auth] local .venv (.env @ {REPO_ROOT})")

fabric_token = cred.get_token(FABRIC_SCOPE).token
headers      = {"Authorization": f"Bearer {fabric_token}"}

ws = next(w for w in requests.get(f"{FABRIC_API}/workspaces", headers=headers).json()["value"]
          if w["displayName"] == WORKSPACE)
ws_id = ws["id"]
db    = next(d for d in requests.get(f"{FABRIC_API}/workspaces/{ws_id}/kqlDatabases", headers=headers).json()["value"]
             if d["displayName"] == KQLDB_NAME)
query_uri = db["properties"]["queryServiceUri"]
print("queryServiceUri:", query_uri)

kcsb  = KustoConnectionStringBuilder.with_azure_token_credential(query_uri, cred)
kusto = KustoClient(kcsb)

[auth] using cached credentials
[auth] local .venv (.env @ C:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection)


queryServiceUri: https://trd-xr97y56tuzzkxy5cgp.z5.kusto.fabric.microsoft.com


## 3. Pull wide data and build sliding windows

In [3]:
cols_kql = ", ".join(SENSORS)
q = f"""
raw_telemetry_wide_mv
| where machine_id == '{MACHINE}'
| where ts_bin > now() - {LOOKBACK}
| order by ts_bin asc
| project ts_bin, {cols_kql}
"""
res = kusto.execute(KQLDB_NAME, q)
df  = dataframe_from_result_table(res.primary_results[0])
print("raw rows from MV:", len(df))
df.head()

raw rows from MV: 1783


,ts_bin,temperature_motor,temperature_bearing,vibration_axial,vibration_radial,current,power,spindle_rpm,pressure_hydraulic
0,2026-05-14 18:18:54+00:00,63.2267,57.4593,0.2462,0.3156,10.5868,7.1005,3135.8071,118.0144
1,2026-05-14 18:18:55+00:00,63.739,58.1429,0.2191,0.3472,11.1928,7.2117,3142.4842,118.8204
2,2026-05-14 18:18:56+00:00,62.625,58.4304,0.1952,0.3073,11.3181,7.2893,3165.3083,116.7948
3,2026-05-14 18:18:57+00:00,63.3241,58.3814,0.1743,0.2515,10.8377,7.5002,3181.6407,118.1155
4,2026-05-14 18:18:58+00:00,62.9421,57.5176,0.1437,0.2117,11.6273,7.4068,3205.15,115.9281


In [4]:
# Forward-fill any NULL cells per sensor (industrial-historian standard).
# Keeps SENSORS ordering, drops still-NaN initial rows (no value yet).
df_ff = df.set_index("ts_bin")[SENSORS].ffill().dropna(how="any")
values = df_ff.to_numpy(dtype=np.float32)
print("after ffill+dropna:", values.shape)

# Sliding windows of WINDOW_SIZE bins, stride 1.
T, F = values.shape
if T < WINDOW_SIZE + 4:
    raise RuntimeError(f"Not enough bins ({T}) to train. Increase LOOKBACK or wait for more data.")
n_win = T - WINDOW_SIZE + 1
X = np.lib.stride_tricks.sliding_window_view(values, (WINDOW_SIZE, F))[:, 0, :, :]
X = np.ascontiguousarray(X.astype(np.float32))
print("training tensor:", X.shape, "(N_windows, window_size, n_features)")

after ffill+dropna: (1783, 8)
training tensor: (1720, 64, 8) (N_windows, window_size, n_features)


## 4. Model + training

In [5]:
import torch
from torch import nn

torch.manual_seed(0)

# --- Per-feature normalization stats (computed on training set only) ---
# Baked into the ONNX model below so the KQL scoring path stays agnostic of
# feature scales. Avoids the previous problem where the loss was dominated
# by RPM (~3100) and other features were ignored.
mean = X.mean(axis=(0, 1))                  # (n_features,)
std  = X.std (axis=(0, 1)) + 1e-8
print("per-feature mean:", np.round(mean, 3))
print("per-feature std :", np.round(std,  3))

X_norm = ((X - mean) / std).astype(np.float32)
t      = torch.from_numpy(X_norm)

class WindowAE(nn.Module):
    def __init__(self, n_features: int, hidden: int = HIDDEN):
        super().__init__()
        self.enc = nn.LSTM(n_features, hidden, batch_first=True)
        self.dec = nn.LSTM(hidden, n_features, batch_first=True)
    def forward(self, x):
        h, _   = self.enc(x)
        out, _ = self.dec(h)
        return out

model    = WindowAE(n_features=F)
opt      = torch.optim.Adam(model.parameters(), lr=LR)
sched    = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn  = nn.MSELoss()
n        = t.shape[0]

# Mini-batch SGD with cosine LR decay.
for epoch in range(EPOCHS):
    perm        = torch.randperm(n)
    epoch_loss  = 0.0
    for i in range(0, n, BATCH):
        idx   = perm[i : i + BATCH]
        batch = t[idx]
        opt.zero_grad()
        e = loss_fn(model(batch), batch)
        e.backward()
        opt.step()
        epoch_loss += e.item() * batch.shape[0]
    epoch_loss /= n
    sched.step()
    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:>3}  loss {epoch_loss:.6f}  lr {sched.get_last_lr()[0]:.2e}")

final_loss = float(epoch_loss)


per-feature mean: [6.161500e+01 5.692900e+01 2.010000e-01 2.940000e-01 1.186300e+01
 7.898000e+00 3.089521e+03 1.197630e+02]
per-feature std : [ 1.473  1.121  0.041  0.06   0.764  0.417 16.611  2.342]


epoch   0  loss 0.789477  lr 1.00e-03


epoch  20  loss 0.140871  lr 8.95e-04


epoch  40  loss 0.133665  lr 6.39e-04


epoch  60  loss 0.130568  lr 3.31e-04


epoch  80  loss 0.129290  lr 8.65e-05


epoch  99  loss 0.129059  lr 0.00e+00


## 5. Export to ONNX (with `ScoreWrapper`) + compute threshold

The wrapper returns a single scalar score per window: MSE averaged over
**both** the time axis and the feature axis. The threshold is computed
from the per-window losses on the training set so the scoring function
in KQL can read it from `metadata.threshold` without a hard-coded value.

In [6]:
class NormalizedScoreWrapper(nn.Module):
    """Bake per-feature (x - mean) / std normalization INTO the ONNX graph.

    The KQL scoring function passes raw values from the MV; this wrapper
    handles normalization, autoencoding, and per-window MSE in one pass.
    Output: (batch,) scalar score per window in normalized feature space.
    """
    def __init__(self, ae: nn.Module, mean_arr, std_arr):
        super().__init__()
        self.ae = ae
        self.register_buffer("mean", torch.as_tensor(mean_arr, dtype=torch.float32))
        self.register_buffer("std",  torch.as_tensor(std_arr,  dtype=torch.float32).clamp_min(1e-8))
    def forward(self, x):
        z     = (x - self.mean) / self.std
        recon = self.ae(z)
        return ((recon - z) ** 2).mean(dim=(1, 2))

wrapped = NormalizedScoreWrapper(model, mean, std).eval()
dummy   = torch.zeros((1, WINDOW_SIZE, F), dtype=torch.float32)

buf = io.BytesIO()
torch.onnx.export(
    wrapped, dummy, buf,
    input_names=["window"], output_names=["score"],
    dynamic_axes={"window": {0: "batch"}, "score": {0: "batch"}},
    opset_version=17,
)
onnx_bytes = buf.getvalue()

# Sandbox onnxruntime supports IR <= 9.
import onnx
mp = onnx.load_from_string(onnx_bytes)
if mp.ir_version > 9:
    mp.ir_version = 9
    onnx_bytes = mp.SerializeToString()

onnx_b64 = base64.b64encode(onnx_bytes).decode("ascii")
print(f"ONNX size: {len(onnx_bytes)/1024:.1f} KB  (base64 {len(onnx_b64)/1024:.1f} KB), ir={mp.ir_version}")


C:\Users\faustopalma\AppData\Local\Temp\ipykernel_42672\838745479.py:22: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0514 20:58:01.822000 42672 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


W0514 20:58:03.208000 42672 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms


W0514 20:58:03.212000 42672 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align


W0514 20:58:03.214000 42672 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `NormalizedScoreWrapper([...]` with `torch.export.export(..., strict=False)`...


...

C:\Users\faustopalma\AppData\Local\Programs\Python\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "C:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection\.venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_ver

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅
ONNX size: 132.4 KB  (base64 176.5 KB), ir=9


In [7]:
# Per-window scores on the training set -> threshold = mean + K*std.
import onnxruntime as ort
sess = ort.InferenceSession(onnx_bytes)
scores_train = sess.run(None, {"window": X})[0]
threshold = float(scores_train.mean() + THRESHOLD_K * scores_train.std())
print(f"train scores: mean={scores_train.mean():.6f}  std={scores_train.std():.6f}  -> threshold={threshold:.6f} (K={THRESHOLD_K})")
print("sample scores:", scores_train[:3])

train scores: mean=0.129059  std=0.064787  -> threshold=0.388205 (K=4.0)
sample scores: [0.89033115 0.8812621  0.87318945]


## 6. Upload into the `models` table

In [8]:
next_v = 1
try:
    res = kusto.execute(KQLDB_NAME, f"models | where name == \'{MODEL_NAME}\' | summarize v = max(version)")
    cur = dataframe_from_result_table(res.primary_results[0])["v"].iloc[0]
    if cur is not None and not pd.isna(cur):
        next_v = int(cur) + 1
except Exception as ex:
    print("warn: lookup version failed, defaulting to 1:", ex)

created_at = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
metadata   = json.dumps({
    "final_loss":     final_loss,
    "n_windows":      int(X.shape[0]),
    "trained_at":     created_at,
    "machine":        MACHINE,
    "sensors":        SENSORS,
    "bin_size":       BIN_SIZE,
    "threshold":      threshold,
    "threshold_k":    THRESHOLD_K,
    "hidden":         HIDDEN,
    "epochs":         EPOCHS,
    "batch_size":     BATCH,
    "lr":             LR,
    # Normalization stats are baked into the ONNX graph; mirrored here
    # for observability/debugging only (KQL scoring does NOT read them).
    "feature_mean":   {s: float(m) for s, m in zip(SENSORS, mean)},
    "feature_std":    {s: float(v) for s, v in zip(SENSORS, std)},
})

sensors_dyn = json.dumps(SENSORS)
cmd = (
    ".set-or-append models <|\n"
    "print "
    f"name=\'{MODEL_NAME}\', "
    f"version=int({next_v}), "
    f"created_at=datetime({created_at}), "
    "framework=\'onnx\', "
    f"window_size=int({WINDOW_SIZE}), "
    f"sensors=todynamic(\'{sensors_dyn}\'), "
    f"payload=\'{onnx_b64}\', "
    f"metadata=todynamic(\'{metadata}\')"
)
kusto.execute_mgmt(KQLDB_NAME, cmd)
print(f"uploaded {MODEL_NAME} v{next_v} (threshold={threshold:.6f})")


uploaded multivariate_ae__M-001 v3 (threshold=0.388205)


## 7. Smoke test in KQL

In [9]:
smoke = f"""
score_multivariate_onnx_batch_from_mv('{MODEL_NAME}', '{MACHINE}')
| top 5 by window_end desc
| project window_start, window_end, score, is_anomaly, model_version
"""
res = kusto.execute(KQLDB_NAME, smoke)
dataframe_from_result_table(res.primary_results[0])

,window_start,window_end,score,is_anomaly,model_version
0,2026-05-14 18:56:34+00:00,2026-05-14 18:57:37+00:00,0.085788,False,3
1,2026-05-14 18:55:30+00:00,2026-05-14 18:56:33+00:00,0.134606,False,3
2,2026-05-14 18:54:26+00:00,2026-05-14 18:55:29+00:00,0.10689,False,3
3,2026-05-14 18:53:22+00:00,2026-05-14 18:54:25+00:00,0.245127,False,3
4,2026-05-14 18:52:18+00:00,2026-05-14 18:53:21+00:00,0.12591,False,3
